# Speaker Verification — ResNet50 + Cosine Triplet Loss

**Cosine Triplet Loss formula:**  
`loss = max( sim(a,n) - sim(a,p) + margin, 0 )`  
The model is pushed to make `sim(anchor, positive) > sim(anchor, negative)` by at least `margin`.

---
## 1. Dataset Setup

In [ ]:
import os
import pandas as pd
import numpy as np

# ── Training data ──────────────────────────────────────────────────────────────
DATASET_ROOT   = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
CSV_PATH       = os.path.join(DATASET_ROOT, "train_pairs.csv")

# ── Test data ──────────────────────────────────────────────────────────────────
TEST_ROOT      = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"
TEST_AUDIO_DIR = os.path.join(TEST_ROOT, "test-clean")
TEST_CSV       = os.path.join(TEST_ROOT, "test_pairs.csv")

print("Train audio:", BASE_AUDIO_DIR)
print("Test audio: ", TEST_AUDIO_DIR)

## 2. Load Training Data

Resolve absolute paths and extract speaker IDs.

In [ ]:
df = pd.read_csv(CSV_PATH)

def to_absolute_path(rel_path):
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

def get_speaker_id(rel_path):
    return rel_path.split("/")[1]   # "train-clean-100/311/..." → "311"

df["path1_abs"]   = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"]   = df["audio_path_2"].apply(to_absolute_path)
df["speaker1_id"] = df["audio_path_1"].apply(get_speaker_id)
df["speaker2_id"] = df["audio_path_2"].apply(get_speaker_id)

print(f"Total rows: {len(df)} | Unique speakers: {df['speaker1_id'].nunique()}")

## 3. Audio Transforms — 5-Second, Random Crop (Training) / Center Crop (Eval)

In [ ]:
import torch
import torchaudio.transforms as T

TARGET_SR     = 16000
TARGET_LENGTH = TARGET_SR * 5   # 48000 samples

def crop_or_pad(audio, is_train=True):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = (np.random.randint(0, length - TARGET_LENGTH) if is_train
                 else (length - TARGET_LENGTH) // 2)
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform   = T.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
amplitude_to_db = T.AmplitudeToDB()
print("Transforms ready.")

## 4. TripletDataset

Each item returns (Anchor, Positive, Negative). Speaker lookup is built from the CSV.

In [ ]:
import soundfile as sf
from collections import defaultdict
from torch.utils.data import Dataset

class TripletDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db

        self.speaker_to_paths = defaultdict(list)
        for _, row in dataframe.iterrows():
            self.speaker_to_paths[row["speaker1_id"]].append(row["path1_abs"])
            self.speaker_to_paths[row["speaker2_id"]].append(row["path2_abs"])

        self.samples = [(p, spk)
                        for spk, paths in self.speaker_to_paths.items()
                        for p in paths]
        self.all_speakers = list(self.speaker_to_paths.keys())

    def __len__(self): return len(self.samples)

    def load_audio(self, path, is_train=True):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio, is_train=is_train)
        audio = torch.tensor(audio).float().unsqueeze(0)
        return self.amplitude_to_db(self.mel_transform(audio))

    def __getitem__(self, idx):
        anchor_path, anchor_spk = self.samples[idx]

        # Positive: same speaker, different file
        pos_candidates = [p for p in self.speaker_to_paths[anchor_spk] if p != anchor_path]
        if not pos_candidates:
            pos_candidates = self.speaker_to_paths[anchor_spk]
        pos_path = pos_candidates[np.random.randint(len(pos_candidates))]

        # Negative: different speaker
        neg_spk = anchor_spk
        while neg_spk == anchor_spk:
            neg_spk = self.all_speakers[np.random.randint(len(self.all_speakers))]
        neg_path = self.speaker_to_paths[neg_spk][np.random.randint(len(self.speaker_to_paths[neg_spk]))]

        return (self.load_audio(anchor_path), self.load_audio(pos_path), self.load_audio(neg_path))

## 5. SpeakerPairDataset (for Test Evaluation)

In [ ]:
class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df              = dataframe
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self): return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio, is_train=False)  # center crop for evaluation
        audio = torch.tensor(audio).float().unsqueeze(0)
        return self.amplitude_to_db(self.mel_transform(audio))

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel1  = self.load_audio(row["path1_abs"])
        mel2  = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

## 6. Initialize Datasets & DataLoaders

In [ ]:
from torch.utils.data import DataLoader

# Training triplet dataset
train_dataset = TripletDataset(df, mel_transform, amplitude_to_db)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
print(f"Train dataset: {len(train_dataset):,} triplets | {len(train_loader):,} batches")

# Test pair dataset
test_df = pd.read_csv(TEST_CSV)
test_df["path1_abs"] = test_df["audio_path_1"].apply(lambda r: os.path.join(TEST_AUDIO_DIR, r))
test_df["path2_abs"] = test_df["audio_path_2"].apply(lambda r: os.path.join(TEST_AUDIO_DIR, r))

test_dataset  = SpeakerPairDataset(test_df, mel_transform, amplitude_to_db)
test_loader   = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f"Test dataset:  {len(test_dataset):,} pairs   | {len(test_loader):,} batches")

## 7. Model Architecture — ResNetSpeaker (ResNet50)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resNet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)   # L2 normalize → cosine = dot product

## 8. Cosine Triplet Loss

`loss = max( sim(anchor, negative) - sim(anchor, positive) + margin, 0 )`

- Same as standard Triplet Loss but uses **Cosine Similarity** instead of Euclidean distance
- `margin = 0.3` — the model must be at least 0.3 more similar to the positive than the negative
- Accuracy: `sim(a, p) > sim(a, n)` — did we rank the same speaker higher?

In [ ]:
class CosineTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        sim_pos = F.cosine_similarity(anchor, positive)   # should be high (→ 1)
        sim_neg = F.cosine_similarity(anchor, negative)   # should be low  (→ 0)
        loss    = torch.clamp(sim_neg - sim_pos + self.margin, min=0.0)
        return loss.mean()

## 9. Training Configuration

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = ResNetSpeaker(embedding_dim=256).to(device)
criterion = CosineTripletLoss(margin=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)
print("Loss:   CosineTripletLoss (margin=0.3)")

## 10. Training Loop — 30 Epochs

- **Triplet accuracy** = % of triplets where `sim(a,p) > sim(a,n)` (should trend toward 1.0)
- Auto-saves checkpoint after every epoch (Kaggle 12-hour protection)

In [ ]:
from tqdm import tqdm

num_epochs  = 30
print_every = 50

loss_history     = []
acc_history      = []
sim_pos_history  = []   # avg cosine(anchor, positive) per epoch
sim_neg_history  = []   # avg cosine(anchor, negative) per epoch

for epoch in range(num_epochs):
    model.train()
    total_loss    = 0
    total_correct = 0
    total_samples = 0
    total_sim_pos = 0
    total_sim_neg = 0

    bar = tqdm(enumerate(train_loader), total=len(train_loader),
               desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (anchor, positive, negative) in bar:
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        # Triplet accuracy using cosine similarity
        with torch.no_grad():
            sim_pos = F.cosine_similarity(emb_a, emb_p)
            sim_neg = F.cosine_similarity(emb_a, emb_n)
            total_correct  += (sim_pos > sim_neg).sum().item()
            total_samples  += anchor.size(0)
            total_sim_pos  += sim_pos.sum().item()
            total_sim_neg  += sim_neg.sum().item()

        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

        if batch_idx % print_every == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    avg_acc  = total_correct / total_samples

    avg_sim_pos = total_sim_pos / total_samples
    avg_sim_neg = total_sim_neg / total_samples

    loss_history.append(avg_loss)
    acc_history.append(avg_acc)
    sim_pos_history.append(avg_sim_pos)
    sim_neg_history.append(avg_sim_neg)

    print(f"\nEpoch [{epoch+1}/{num_epochs}] — Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f} | Sim+(pos): {avg_sim_pos:.4f} | Sim-(neg): {avg_sim_neg:.4f}\n")

    torch.save({
        "model_state":     model.state_dict(),
        "epoch":           epoch,
        "optimizer_state": optimizer.state_dict(),
        "loss_history":    loss_history,
        "acc_history":     acc_history,
        "sim_pos_history": sim_pos_history,
        "sim_neg_history": sim_neg_history,
    }, "checkpoint_resnet50_cosine_triplet.pth")
    print(f"  Checkpoint saved → epoch {epoch+1}")

print("\nTraining complete!")

## 11. Final Model Save

In [ ]:
torch.save({
    "model_state":     model.state_dict(),
    "epoch":           num_epochs - 1,
    "optimizer_state": optimizer.state_dict(),
    "loss_history":    loss_history,
    "acc_history":     acc_history,
    "sim_pos_history": sim_pos_history,
    "sim_neg_history": sim_neg_history,
}, "checkpoint_resnet50_cosine_triplet.pth")
print("Final model saved → checkpoint_resnet50_cosine_triplet.pth")

## 12. Training History Graphs — Loss & Accuracy per Epoch

## 12b. Cosine Similarity Trends — Positive vs Negative per Epoch

Shows whether `sim(anchor, positive)` is rising and `sim(anchor, negative)` is falling over training. A clear divergence means the model is learning.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ResNet34 + Cosine Triplet Loss — Full Training Overview', fontsize=13, fontweight='bold')

# Loss
axes[0, 0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0, 0].set_title('Cosine Triplet Loss per Epoch')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Triplet accuracy
axes[0, 1].plot(epochs_range, acc_history, marker='o', color='steelblue', linewidth=2)
axes[0, 1].set_title('Triplet Accuracy per Epoch\n(sim_pos > sim_neg)')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim(0, 1); axes[0, 1].grid(True, alpha=0.3)

# Positive vs Negative similarity divergence
axes[1, 0].plot(epochs_range, sim_pos_history, marker='o', color='steelblue', linewidth=2, label='Avg sim(anchor, positive)')
axes[1, 0].plot(epochs_range, sim_neg_history, marker='o', color='tomato',    linewidth=2, label='Avg sim(anchor, negative)')
axes[1, 0].fill_between(epochs_range, sim_pos_history, sim_neg_history, alpha=0.15, color='green')
axes[1, 0].set_title('Cosine Similarity Divergence per Epoch')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Cosine Similarity')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

# Similarity margin (pos - neg) per epoch
margin_history = [p - n for p, n in zip(sim_pos_history, sim_neg_history)]
axes[1, 1].plot(epochs_range, margin_history, marker='o', color='green', linewidth=2)
axes[1, 1].axhline(y=0.3, color='gray', linestyle='--', linewidth=1.2, label='Loss margin (0.3)')
axes[1, 1].set_title('Avg Similarity Margin per Epoch\n(sim_pos - sim_neg)')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Margin')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_overview_cosine_triplet.png', dpi=150)
plt.show()
print('Saved → training_overview_cosine_triplet.png')

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ResNet34 + Cosine Triplet Loss — Training Metrics", fontsize=13, fontweight='bold')

axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0].set_title("Cosine Triplet Loss per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, acc_history, marker='o', color='steelblue', linewidth=2)
axes[1].set_title("Triplet Accuracy per Epoch
(sim_pos > sim_neg)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics_cosine_triplet.png", dpi=150)
plt.show()
print("Saved → training_metrics_cosine_triplet.png")

## 13. Pair Evaluation Function

Evaluates on a **pair dataset** (label 0/1) using cosine similarity > 0.5.

In [ ]:
def evaluate_pairs(model, loader, device, label_name="Set"):
    model.eval()
    correct = 0; total = 0
    same_sims, diff_sims = [], []

    with torch.no_grad():
        for mel1, mel2, labels in tqdm(loader, desc=f"Evaluating {label_name}"):
            mel1   = mel1.to(device)
            mel2   = mel2.to(device)
            labels = labels.to(device)

            emb1 = model(mel1)
            emb2 = model(mel2)

            sim   = F.cosine_similarity(emb1, emb2)
            preds = (sim > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            for s, lbl in zip(sim.cpu().tolist(), labels.cpu().tolist()):
                (same_sims if lbl == 1 else diff_sims).append(s)

    acc = correct / total
    print(f"{label_name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    return acc, same_sims, diff_sims

## 14. Evaluate on Training Pairs & Test Set

In [ ]:
# Build training pair loader for evaluation (center crop, no shuffle)
train_pair_df = df.copy()
train_pair_dataset = SpeakerPairDataset(train_pair_df, mel_transform, amplitude_to_db)
train_pair_loader  = DataLoader(train_pair_dataset, batch_size=16, shuffle=False,
                                num_workers=2, pin_memory=True)

train_acc, tr_same_sim, tr_diff_sim = evaluate_pairs(model, train_pair_loader, device, "Training Set")
test_acc,  te_same_sim, te_diff_sim = evaluate_pairs(model, test_loader,       device, "Test Set")

print(f"\nTrain Acc: {train_acc*100:.2f}%  |  Test Acc: {test_acc*100:.2f}%")
print(f"Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")

## 15. Accuracy Comparison Bar Chart

In [ ]:
plt.figure(figsize=(7, 5))
bars = plt.bar(["Training Accuracy", "Test Accuracy"],
               [train_acc, test_acc],
               color=["steelblue", "darkorange"], width=0.4, edgecolor='white')
for bar, val in zip(bars, [train_acc, test_acc]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val*100:.2f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("ResNet34 + Cosine Triplet — Training vs Test Accuracy", fontsize=12, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_comparison_cosine_triplet.png", dpi=150)
plt.show()
print("Saved → accuracy_comparison_cosine_triplet.png")

## 16. Cosine Similarity Distribution — Training vs Test Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Cosine Similarity Distribution — Training vs Test", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], tr_same_sim[:5000], tr_diff_sim[:5000], "Training Set"),
    (axes[1], te_same_sim[:5000], te_diff_sim[:5000], "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
    ax.set_title(title)
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Count")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cosine_sim_dist_cosine_triplet.png", dpi=150)
plt.show()
print("Saved → cosine_sim_dist_cosine_triplet.png")

## 17. Final Summary

In [ ]:
print("=" * 47)
print("  MODEL:  ResNet34 + Cosine Triplet Loss")
print("-" * 47)
print(f"  Training Accuracy : {train_acc*100:.2f}%")
print(f"  Test Accuracy     : {test_acc*100:.2f}%")
print(f"  Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")
print("=" * 47)
print("Saved files:")
print("  checkpoint_resnet34_cosine_triplet.pth")
print("  training_metrics_cosine_triplet.png")
print("  accuracy_comparison_cosine_triplet.png")
print("  cosine_sim_dist_cosine_triplet.png")